# AIGC Detector — Training on Colab

Runs `train.py` against the real dataset on a Colab GPU. CPU-only training locally is impractical (90k+ images, frozen CLIP forward pass every step).

**Before running:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** (or better) → Save.

Checkpoints are written to Google Drive (not Colab's ephemeral local disk), so a disconnect doesn't lose progress — `train.py` auto-resumes from the latest checkpoint (`resume_from_latest: true` in `configs/train.yaml`) the next time this notebook runs.

**If you restart the kernel (Runtime → Restart session) or reconnect**, cells below that run `!python ...` or `import train`/`import evaluate` need `/content/choochoo` as the working directory again — each code cell from step 6 onward starts with `%cd /content/choochoo` for exactly this reason, so re-running any single cell out of order is safe.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — set Runtime > Change runtime type > GPU")

## 1. Mount Google Drive

Used for checkpoint persistence across session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set secrets

`train.py` only needs `HF_TOKEN` (for the CLIP backbone download — works without it, just rate-limited as an anonymous request). Add it via Colab's Secrets manager: left sidebar → key icon → **New secret** → name `HF_TOKEN`, paste a token from [huggingface.co](https://huggingface.co) (account settings → Access Tokens) → toggle **Notebook access** on.

(`KAGGLE_USERNAME`/`KAGGLE_KEY` and `WANDB_API_KEY` are NOT needed here — they're only used by `data/prepare_datasets.py`, which you don't need to run since the dataset is already committed in the repo.)

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set.")
except Exception as e:
    print("No HF_TOKEN secret configured — continuing without one (downloads will be rate-limited, not blocked).")

## 3. Clone the repo

Pulls whatever is currently pushed to GitHub — make sure your local commits are pushed before running this. Safe to re-run after a full runtime reset (local disk wiped); if the folder already exists from earlier in the same session, this just skips re-cloning.

In [ ]:
!test -d /content/choochoo || git clone https://github.com/windyheng/choochoo.git /content/choochoo
%cd /content/choochoo

## 4. Install dependencies

In [ ]:
%cd /content/choochoo
!pip install -q -r requirements.txt

## 5. Point checkpoints and results at Drive

Symlinks `checkpoints/` (what `configs/train.yaml`'s `checkpoint_dir` points to) and `results/` into Drive, so both survive a session reset — `results/` holds the robustness table and error-analysis thumbnails from the evaluation step below, which you don't want to lose to a disconnect any more than the checkpoints.

In [ ]:
%cd /content/choochoo
!mkdir -p /content/drive/MyDrive/choochoo_checkpoints
!rm -rf checkpoints
!ln -sfn /content/drive/MyDrive/choochoo_checkpoints checkpoints
!ls -la checkpoints

!mkdir -p /content/drive/MyDrive/choochoo_results
!rm -rf results
!ln -sfn /content/drive/MyDrive/choochoo_results results
!ls -la results

## 6. Train

Re-run this same cell after a disconnect/reconnect (remount Drive first, and re-run step 5 so the checkpoint symlink exists again) — it resumes automatically from the latest checkpoint.

In [ ]:
%cd /content/choochoo
!python train.py --config configs/train.yaml

## (Optional) Train the ablation branches

`clip_only` and `artifact_only` are separately-trained models (different `FusionHead` shape each) — needed for `evaluate.py`'s branch-comparison ablation. Each needs its own `checkpoint_dir`; easiest is a separate Drive folder per branch.

In [ ]:
# %cd /content/choochoo
# !mkdir -p /content/drive/MyDrive/choochoo_checkpoints_clip_only
# !rm -rf checkpoints && ln -sfn /content/drive/MyDrive/choochoo_checkpoints_clip_only checkpoints
# !python train.py --config configs/train.yaml --branch clip_only

# !mkdir -p /content/drive/MyDrive/choochoo_checkpoints_artifact_only
# !rm -rf checkpoints && ln -sfn /content/drive/MyDrive/choochoo_checkpoints_artifact_only checkpoints
# !python train.py --config configs/train.yaml --branch artifact_only

## 7. Evaluate the trained checkpoint

Run this once training has produced at least one checkpoint (doesn't need to be fully finished — even a partially-trained checkpoint is enough to smoke-test the pipeline). If you trained the `full` branch above, `checkpoints/` still points at that Drive folder, so the cell below finds the latest one automatically rather than requiring you to know its exact filename.

In [ ]:
%cd /content/choochoo
from train import find_latest_checkpoint

# Reads straight from Drive rather than through the local checkpoints/
# symlink — the symlink can end up stale/missing if step 5 wasn't re-run
# after a kernel restart or reconnect, since %cd/symlink state doesn't
# survive a restarted kernel even though the repo files on disk might.
CHECKPOINT_DRIVE_DIR = "/content/drive/MyDrive/choochoo_checkpoints"
ckpt = find_latest_checkpoint(CHECKPOINT_DRIVE_DIR)

if ckpt is None:
    print("No checkpoint found. Diagnostics:")
    !pwd
    !ls -la /content/choochoo 2>&1 | head -5
    !ls -la {CHECKPOINT_DRIVE_DIR} 2>&1
    raise AssertionError(
        "no checkpoint found under " + CHECKPOINT_DRIVE_DIR + " — run the train cell first, "
        "or check the diagnostics above (is Drive actually mounted to this account/folder?)"
    )
print("Using checkpoint:", ckpt)

### 7a. Smoke test first

`evaluate.py` runs the full robustness matrix (~16 conditions × the whole
test set — 19,410 images here, so ~310k unbatched forward passes) and
prints nothing until it's completely done. Before committing to that
multi-hour run, prove the pipeline actually works end-to-end against a tiny
subset — finishes in well under a minute.

In [ ]:
%cd /content/choochoo
!head -50 data/cache/splits/test.csv > /tmp/test_small.csv
# evaluate.py now resumes by skipping conditions already in --out's file, and
# defaults --predictions_out to results/predictions.csv (shared with the
# full run in 7b) — force fresh /tmp outputs each time so re-running this
# after further training actually re-scores instead of reusing a stale
# result, and so this smoke test never pollutes the full run's real output.
!rm -f /tmp/robustness_table_smoke.csv /tmp/predictions_smoke.csv
!python evaluate.py --checkpoint {ckpt} --eval_csv /tmp/test_small.csv --out /tmp/robustness_table_smoke.csv --predictions_out /tmp/predictions_smoke.csv
!cat /tmp/robustness_table_smoke.csv

### 7b. Full run

Only run this once the smoke test above succeeds. This is the multi-hour one — consider kicking it off right before stepping away.

**Resumable**: `evaluate.py` now writes each condition's results to `results/robustness_table.csv`/`results/predictions.csv` as soon as that condition finishes, and skips conditions already recorded there. Since `results/` is symlinked to Drive (step 5), if Colab disconnects mid-run, just remount Drive and re-run this *exact same cell* — it picks up from the next unfinished condition instead of starting over from scratch.

**Caveat**: this only checks *which conditions* are recorded, not which checkpoint produced them. If you train further and get a newer checkpoint (updated `ckpt` from step 7), re-running this cell will skip conditions already in `results/robustness_table.csv` even though they're now stale — delete `results/robustness_table.csv` and `results/predictions.csv` first if you want a genuinely fresh run against the new checkpoint, rather than a resume of the old one.

In [ ]:
%cd /content/choochoo
!python evaluate.py --checkpoint {ckpt} --eval_csv data/cache/splits/test.csv --out results/robustness_table.csv

## 8. Error analysis

Mines representative false positives/negatives from the predictions the evaluate step above just wrote, bucketed by transform condition, into `results/error_analysis/` (thumbnail grids + `error_summary.csv`).

In [ ]:
%cd /content/choochoo
!python error_analysis.py --predictions results/predictions.csv --out results/error_analysis/

## 9. Required deliverable: infer.py smoke test

Runs the actual required inference script against a handful of real sample images, to confirm the checkpoint produces sane, varied predictions (not everything collapsed to the same score).

In [ ]:
%cd /content/choochoo
!mkdir -p /tmp/infer_smoke_test
!find data/raw -type f -name '*.jpg' | head -5 | xargs -I{} cp {} /tmp/infer_smoke_test/
!python infer.py --input_dir /tmp/infer_smoke_test --out results/smoke_test_preds.json --checkpoint {ckpt}
!cat results/smoke_test_preds.json

## 10. Pull results back down

`results/` is symlinked to Drive (step 5), so everything above (`robustness_table.csv`, `predictions.csv`, `error_analysis/`, `smoke_test_preds.json`) is already persisted there — download the whole `choochoo_results` Drive folder to your machine, or `git add results/ && git commit` from this Colab session directly (need `git config user.email`/`user.name` and a way to auth/push, e.g. a GitHub token) if you'd rather commit from here.